# The positions test on a small open model

Runtime → Change runtime type → T4 GPU. The first run downloads Qwen2.5-3B-Instruct (about 6 GB).

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
!pip -q install -U transformers accelerate
import huggingface_hub.constants
huggingface_hub.constants.HF_HUB_DISABLE_XET = True

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to("cuda").eval()
QUESTION = "{a} owes {b} fifty dollars. Who has to pay the money? Answer with one name."


@torch.no_grad()
def run(text, right, wrong, no_positions=False):
    p = tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    kw = {"position_ids": torch.zeros_like(ids)} if no_positions else {}   # every word at position 0
    out = model(ids, output_hidden_states=True, **kw)
    r, w = tok(right, add_special_tokens=False).input_ids[0], tok(wrong, add_special_tokens=False).input_ids[0]
    lens = []                                                          # logit lens: read every block's output
    for i, h in enumerate(out.hidden_states):
        x = h[0, -1] if i == len(out.hidden_states) - 1 else model.model.norm(h[0, -1])
        lg = model.lm_head(x).float()
        lens.append((tok.decode([int(lg.argmax())]), float(torch.softmax(lg, -1)[r])))
    final = out.logits[0, -1].float()
    reply = None
    if not no_positions:
        reply = tok.decode(model.generate(ids, max_new_tokens=8, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True).strip()
    return bool(final[r] > final[w]), reply, lens

## Both ways round

In [ ]:
for a, b in [("Tom", "Ana"), ("Ana", "Tom")]:
    print(QUESTION.format(a=a, b=b), "->", run(QUESTION.format(a=a, b=b), a, b)[1])

## Logit lens: the best guess and the right name's chance after each block

In [ ]:
_, _, lens = run(QUESTION.format(a="Tom", b="Ana"), "Tom", "Ana")
for i, (top, p) in enumerate(lens):
    print(f"block {i:2}  {p:6.1%}  {top!r}")

## 12 name pairs, both ways round: normal vs every word at position 0

Right = the model rates the right name above the wrong one.

In [ ]:
PAIRS = [("Tom", "Ana"), ("Sam", "Lily"), ("Omar", "Sara"), ("Ben", "Mia"), ("Leo", "Zoe"), ("Max", "Eva"),
         ("Jack", "Emma"), ("Adam", "Nina"), ("Paul", "Rosa"), ("Dan", "Kate"), ("Ali", "Lena"), ("Tim", "Julia")]
for no_positions in (False, True):
    right = sum(run(QUESTION.format(a=x, b=y), x, y, no_positions)[0] for a, b in PAIRS for x, y in [(a, b), (b, a)])
    print(f"every word at position 0: {no_positions!s:5}  ->  right {right} of {2 * len(PAIRS)}")

## Try your own

In [ ]:
text, right, wrong = "Lina lent Sami her bike. Who has to give the bike back? Answer with one name.", "Sami", "Lina"
print(run(text, right, wrong)[:2])